# 🏠 German Rental Price Classification

**Dataset:** `immo_data.csv` (German rental listings scraped from ImmoScout24)
**Task:** Multi-class classification
**Method:** CRISP-DM (see `PROJECT_ROADMAP.md`)

---

## 1. Business Understanding

### 1.1 Background
Rent prices in Germany vary widely between regions, cities and types of property. A tenant browsing hundreds of listings finds it hard to judge whether a specific offer is **cheap, fair, or expensive** for what it offers, and a rental platform has no automatic way to tell them.

### 1.2 Problem (draft)
- **Main user:** a rental platform (e.g., ImmoScout24) and, through it, the tenants who use it.
- **What the model does:** predict the **expected** rent class (Low / Medium / High per m²) of a flat from its attributes (location, size, age, condition, equipment…), **without seeing its price**.
- **Decision it supports:** the platform compares the predicted class with the flat's **actual** price:
  - Predicted *Medium*, actually *High* → ⚠️ **possibly overpriced**
  - Predicted *High*, actually *Medium* → 💎 **good deal**

  This helps tenants judge whether an offer is fair.
- **Secondary goal:** find **which attributes drive** a flat into a higher price class.

### 1.3 Target definition
- **Target (decision attribute):** `rentClass` ∈ {Low, Medium, High}
- **Built from:** `rentPerSqm = baseRent ÷ livingSpace`
- **Why per m² and not total rent?** Total rent mainly reflects the flat's **size**. Rent per m² removes the size effect and shows the true **price level**, so small and large flats can be compared fairly.

  | Flat | Size | Total rent | Rent per m² |
  |---|---|---|---|
  | A | 30 m² | €600 | **€20/m²** (expensive) |
  | B | 120 m² | €1,200 | **€10/m²** (cheaper) |

- **How the classes are made:** sort all flats by rent per m² and cut the list into **3 equal-sized groups** (tertiles). Exact cut-off values will be computed in Phase 5.
- **Leakage warning:** since the target is built from rent, every rent-related column (`baseRent`, `totalRent`, `baseRentRange`, `serviceCharge`…) must be **removed from the model inputs**. Otherwise the model would be reading the answer.

### 1.4 Success criteria (draft, to be verified in Phase 5.4)
| Item | Value | Reasoning |
|---|---|---|
| **Assumption** | Classes ≈ balanced (~33% each) | The classes are created by splitting into 3 equal groups |
| **Baseline** | ≈ 33% accuracy | A "no-brain" model that always predicts one class gets 1 in 3 right |
| **Main goal** | **Accuracy ≥ 70%** | Clearly beats the baseline; "good" range for a balanced 3-class problem |
| **Mistake priority** | **Far mistakes (Low ↔ High) must be rare** | The classes are *ordered*: a near mistake (High↔Medium) gives a mild misjudgement, but a far mistake (High↔Low) would put a strongly wrong "overpriced" or "good deal" label on a flat and mislead tenants |
| **Also reported** | Precision / recall / F1 per class, balanced accuracy, confusion matrix | To make sure no single class is predicted badly |

> **Why accuracy is acceptable here:** it is a valid main measure only when (1) the classes are balanced **and** (2) mistakes cost about the same. (1) is true by design. (2) is only partly true, so we add a confusion-matrix check on far mistakes.
>
> If Phase 5.4 shows the classes are *not* balanced, switch the main goal to **balanced accuracy / macro F1**.


In [39]:
import sys

print(sys.executable)

c:\Users\hanan\Desktop\dataSetproject1\.venv\Scripts\python.exe


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Data Understanding

In [41]:
path = "../data/raw/immo_data.csv"
df=pd.read_csv(path,low_memory=False)
pd.set_option('display.max_columns', None)
print(df.shape)
print(df.info())
display(df.head(5))



KeyboardInterrupt: 

## Data dictionary

In [ ]:
summary=pd.DataFrame({"dtype":df.dtypes,
                      "non_null":df.notna().sum(),
                      "missing_%":(df.isna().mean()*100).round(1),
                      "unique":df.nunique(),
                      "example":df.iloc[0]})

summary.sort_values(by="missing_%",ascending=False)

,dtype,non_null,missing_%,unique,example
telekomHybridUploadSpeed,float64,45020,83.3,1,NaN
electricityBasePrice,float64,46846,82.6,2,NaN
electricityKwhPrice,float64,46846,82.6,15,NaN
energyEfficiencyClass,str,77787,71.1,10,NaN
lastRefurbish,float64,80711,70.0,88,NaN
heatingCosts,float64,85518,68.2,5669,NaN
noParkSpaces,float64,93052,65.4,71,1.0
petsAllowed,str,154277,42.6,3,NaN
interiorQual,str,156185,41.9,4,normal
thermalChar,float64,162344,39.6,7847,181.4


### 2.5 Column types

In [ ]:
# --- 1. Decisions that need human knowledge -------------------------------
id_address = ["scoutId", "houseNumber", "street", "streetPlain"]  # identify ONE listing/place
free_text  = ["description", "facilities"]                        # long sentences
codes      = ["geo_plz",                                          # postcode: a label, not a quantity
              "baseRentRange", "livingSpaceRange",                # "...Range" = group number (1, 2, 3...)
              "noRoomsRange", "yearConstructedRange"]
date_cols  = ["date"]

# --- 2. A function that decides the group of ONE column -------------------
def column_group(col):
    n = df[col].nunique()                          # number of different values
    if col in id_address: return "ID / address"
    if col in free_text:  return "Text (free)"
    if col in codes:      return "Categorical (code)"
    if col in date_cols:  return "Date"
    if n == 1:            return "Constant"         # same value in every row
    if pd.api.types.is_bool_dtype(df[col]):    return "Boolean"
    if pd.api.types.is_numeric_dtype(df[col]): return "Numeric"
    return "Categorical (text)"                     # anything left = text labels

# --- 3. Apply it to every column and save the result in the summary -------
# adds a new column to the summary DataFrame, called "group", which contains the group of each column
summary["group"] = [column_group(col) for col in summary.index]

# --- 4. Print each group with its columns ----------------------------------
# .groupby("group") creates a new DataFrame for each group, and .groups.items() returns a list of tuples (group_name, column_names)
for group, cols in summary.groupby("group").groups.items():
    print(f"{group} ({len(cols)}): {list(cols)}\n")

Boolean (6): ['newlyConst', 'balcony', 'hasKitchen', 'cellar', 'lift', 'garden']

Categorical (code) (5): ['yearConstructedRange', 'baseRentRange', 'geo_plz', 'noRoomsRange', 'livingSpaceRange']

Categorical (text) (13): ['regio1', 'heatingType', 'telekomTvOffer', 'firingTypes', 'geo_bln', 'geo_krs', 'condition', 'interiorQual', 'petsAllowed', 'typeOfFlat', 'regio2', 'regio3', 'energyEfficiencyClass']

Constant (1): ['telekomHybridUploadSpeed']

Date (1): ['date']

ID / address (4): ['scoutId', 'houseNumber', 'street', 'streetPlain']

Numeric (17): ['serviceCharge', 'picturecount', 'pricetrend', 'telekomUploadSpeed', 'totalRent', 'yearConstructed', 'noParkSpaces', 'baseRent', 'livingSpace', 'noRooms', 'thermalChar', 'floor', 'numberOfFloors', 'heatingCosts', 'lastRefurbish', 'electricityBasePrice', 'electricityKwhPrice']

Text (free) (2): ['description', 'facilities']



### 2.6 Duplicates and redundant columns

In [ ]:
n_dup_rows=df.duplicated().sum()
n_dup_ids=df["scoutId"].duplicated().sum()

# in case you want to remove duplicates, uncomment the following lines:
# before = len(df)
# df = df.drop_duplicates()            # keeps the FIRST copy, removes the rest
# print("Removed:", before - len(df), "duplicate rows")
print("Fully duplicated rows:", n_dup_rows)
print("Duplicated scoutIds:  ", n_dup_ids)

Fully duplicated rows: 0
Duplicated scoutIds:   0


In [ ]:
# from head "regio1", "geo_bln" and "regio2", "geo_krs" looks like they are the same, but with different names. Let's check if they are really the same:
pairs=[("regio1", "geo_bln"), ("regio2", "geo_krs")]

for a,b in pairs:
    same=(df[a]==df[b]).mean()
    print(f"{a} vs {b}: identical in {same:.1%} of rows")

regio1 vs geo_bln: identical in 100.0% of rows
regio2 vs geo_krs: identical in 100.0% of rows


### 2.7 Leakage check
Leakage = an input column that contains the answer (the price), directly or indirectly.
Rule: "Could I know this value without knowing the flat's rent?" If not, it must not be a model input.


In [ ]:
leakage_cols = {
    "baseRent":      "IS the rent (used to build the target)",
    "totalRent":     "rent + extra costs, so it contains the rent",
    "baseRentRange": "the rent grouped into bands (1 = cheapest ... 9 = most expensive)",
    "serviceCharge": "extra monthly costs set together with the rent, so it is price information",
    "heatingCosts":  "monthly heating cost, also price information",
}

# Print the list so it is documented in the notebook
for col, reason in leakage_cols.items():
    print(f"- {col}: {reason}")

# Save just the names in a list, so we can use it later
leakage_list = list(leakage_cols.keys())
print("\nColumns to exclude from model inputs:", leakage_list)

- baseRent: IS the rent (used to build the target)
- totalRent: rent + extra costs, so it contains the rent
- baseRentRange: the rent grouped into bands (1 = cheapest ... 9 = most expensive)
- serviceCharge: extra monthly costs set together with the rent, so it is price information
- heatingCosts: monthly heating cost, also price information

Columns to exclude from model inputs: ['baseRent', 'totalRent', 'baseRentRange', 'serviceCharge', 'heatingCosts']


### 2.8 Phase 2 summary
- **Size:** 268,850 listings × 49 columns.
- **Duplicates:** none (0 duplicated rows, 0 duplicated scoutIds).
- **Redundant pairs:** `regio1` = `geo_bln` (state) and `regio2` = `geo_krs` (district) are 100% identical, so only one of each is needed.
- **Constant column:** `telekomHybridUploadSpeed` has a single value, so it carries no information.
- **Not usable as-is:** IDs/addresses (`scoutId`, `houseNumber`, `street`, `streetPlain`), free text (`description`, `facilities`).
- **Codes stored as numbers:** `geo_plz` and the `...Range` columns.
- **Leakage:** `baseRent`, `totalRent`, `baseRentRange`, `serviceCharge`, `heatingCosts` must not be model inputs.

## 3. Data Quality Assessment
### 3.1 Missing values

In [ ]:
# pannde series of missing values, in percent, rounded to 1 decimal place
missing=(df.isna().mean()*100).round(1)  


# order the missing values from highest to lowest
# we use missing[missing>0] to remove the percent of cols with no missing val
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:", len(missing), "of", df.shape[1])
missing

Columns with missing values: 27 of 49


telekomHybridUploadSpeed   83.3
electricityBasePrice       82.6
electricityKwhPrice        82.6
energyEfficiencyClass      71.1
lastRefurbish              70.0
heatingCosts               68.2
noParkSpaces               65.4
petsAllowed                42.6
interiorQual               41.9
thermalChar                39.6
numberOfFloors             36.4
streetPlain                26.4
houseNumber                26.4
condition                  25.5
yearConstructed            21.2
firingTypes                21.2
yearConstructedRange       21.2
facilities                 19.7
floor                      19.1
heatingType                16.7
totalRent                  15.1
typeOfFlat                 13.6
telekomUploadSpeed         12.4
telekomTvOffer             12.1
description                 7.3
serviceCharge               2.6
pricetrend                  0.7
dtype: float64

### 3.2 Missing-value plan
The method is chosen **per column**, based on **why** the values are missing. In this step we only *decide*; the actual filling happens in Phase 4 (and inside the model pipeline in Phase 7).

| Option | When to use it | Why |
|---|---|---|
| **Drop** | Mostly empty (> 60%), or the column is removed anyway (leakage, IDs, free text, not about the flat) | Filling a mostly-empty column would mean *inventing* most of its values |
| **Flag (yes/no)** | The gap itself carries meaning | Predicting a value here would invent false facts (e.g. parking that doesn't exist) |
| **"unknown"** | Categories | A missing value may itself be a clue (e.g. a landlord not stating the interior quality) |
| **Model-based** | Numbers with moderate gaps | Predicts each flat's value from its other columns, so it's more precise than one value for all. Compared against median in Phase 8 |
| **Median** | Numbers with very few gaps | Robust to outliers (the mean is not) |

**Assumptions checked with data:**
- `noParkSpaces` contains both empty cells **and** the value 0 (3,855 flats), so an empty cell means "not stated", not "no parking". The flag is therefore defined as **"≥ 1 parking space advertised"**.
- `lastRefurbish` contains impossible years (e.g. 1015, 2919). These are handled in step 3.4.

> ⚠️ The model-based imputer must learn **only from the training data** (never the test data). Otherwise test information leaks in. It will be placed inside a pipeline in Phase 7.


In [ ]:
# The plan: column name → what we will do with it (applied in Phase 4 / Phase 7)
missing_plan = {
    # --- Drop: mostly empty (>60%) ---------------------------------------------
    "telekomHybridUploadSpeed": "drop",   # 83% missing, and constant (1 value)
    "electricityBasePrice":     "drop",   # 83% missing, about electricity, not the flat
    "electricityKwhPrice":      "drop",   # 83% missing, about electricity, not the flat
    "energyEfficiencyClass":    "drop",   # 71% missing

    # --- Drop: removed anyway, so filling is pointless ----------------------------
    "heatingCosts":         "drop",       # leakage (price information)
    "totalRent":            "drop",       # leakage (contains the rent)
    "serviceCharge":        "drop",       # leakage (price information)
    "houseNumber":          "drop",       # address detail
    "streetPlain":          "drop",       # address detail
    "description":          "drop",       # free text
    "facilities":           "drop",       # free text
    "yearConstructedRange": "drop",       # redundant: yearConstructed in bands
    "firingTypes":          "drop",       # messy (132 combined values), overlaps with heatingType
    "telekomTvOffer":       "drop",       # internet-provider offer, not a property of the flat
    "telekomUploadSpeed":   "drop",       # internet-provider offer, not a property of the flat

    # --- Flag (yes/no): the gap itself carries meaning -------------------------
    "lastRefurbish": "flag",              # → wasRefurbished: a year is given = refurbished
    "noParkSpaces":  "flag",              # → hasParking: ≥1 space advertised (empty AND 0 both exist)

    # --- Categories: fill with "unknown" (a missing value may be a clue) -------
    "petsAllowed":  "unknown",
    "interiorQual": "unknown",
    "condition":    "unknown",
    "heatingType":  "unknown",
    "typeOfFlat":   "unknown",

    # --- Numbers with moderate gaps: predict from other columns ----------------
    # (model-based imputation, compared against median in Phase 8)
    "yearConstructed": "model",           # 21% missing
    "floor":           "model",           # 19% missing
    "numberOfFloors":  "model",           # 36% missing
    "thermalChar":     "model",           # 40% missing

    # --- Numbers with very few gaps: median (robust to outliers) ---------------
    "pricetrend": "median",               # 0.7% missing
}

# Safety check 1: does every column with missing values have a decision?
forgotten = [col for col in missing.index if col not in missing_plan]
print("Columns without a decision:", forgotten)

# Safety check 2: does the plan mention a column that doesn't exist (typo)?
typos = [col for col in missing_plan if col not in df.columns]
print("Unknown column names in plan:", typos)

# Count how many columns get each decision
pd.Series(missing_plan).value_counts()


Columns without a decision: []
Unknown column names in plan: []


drop       15
unknown     5
model       4
flag        2
median      1
Name: count, dtype: int64

### 3.3 Summary statistics

What
One table that shows, for every number column: how many values it has, the typical value, the spread, and the smallest and largest values.

Why
It's the fastest way to spot problems. You don't need to look at 268,850 rows; you look at the edges (min and max) and the middle (median). If a number makes no sense in real life, there's a data problem.

In [ ]:
# Show big numbers normally (e.g. 9,999,999.0) instead of scientific notation (1e+07)
pd.set_option("display.float_format", "{:,.1f}".format)


# go over the the colms and check which ones are numeric, and save them in a list called num_cols
num_cols=summary.index[summary["group"]=="Numeric"]

# describe() calculates the statistics; .T flips the table so each column becomes a row
stats = df[num_cols].describe().T
stats



# Row	Meaning
# count	How many rows have a value (not empty)
# mean	Average
# std	Standard deviation: how spread out the values are
# min	Smallest value
# 25%	A quarter of flats are below this
# 50%	The median, the middle flat
# 75%	Three quarters of flats are below this
# max	Largest value


,count,mean,std,min,25%,50%,75%,max
serviceCharge,"261,941.0",151.2,308.3,0.0,95.0,135.0,190.0,"146,118.0"
picturecount,"268,850.0",9.8,6.4,0.0,6.0,9.0,13.0,121.0
pricetrend,"267,018.0",3.4,2.0,-12.3,2.0,3.4,4.6,14.9
telekomUploadSpeed,"235,492.0",28.8,16.3,1.0,10.0,40.0,40.0,100.0
totalRent,"228,333.0",901.3,"33,238.3",0.0,469.8,650.0,985.0,"15,751,535.0"
yearConstructed,"211,805.0","1,966.4",47.0,"1,000.0","1,950.0","1,973.0","1,996.0","2,090.0"
noParkSpaces,"93,052.0",1.3,8.4,0.0,1.0,1.0,1.0,"2,241.0"
baseRent,"268,850.0",694.1,"19,536.0",0.0,338.0,490.0,799.0,"9,999,999.0"
livingSpace,"268,850.0",74.4,254.8,0.0,54.0,67.3,87.0,"111,111.0"
noRooms,"268,850.0",2.6,2.6,1.0,2.0,3.0,3.0,"1,000.0"


In [ ]:
# Count the flats that break each rule
print("Rent is 0 or below:            ", (df["baseRent"] <= 0).sum())
print("Rent above 10,000 €:           ", (df["baseRent"] > 10000).sum())
print("Size below 10 m²:              ", (df["livingSpace"] < 10).sum())
print("Size above 400 m²:             ", (df["livingSpace"] > 400).sum())
print("More than 20 rooms:            ", (df["noRooms"] > 20).sum())
print("Built before 1800 or after 2021:", ((df["yearConstructed"] < 1800) | (df["yearConstructed"] > 2021)).sum())
print("Refurbished after 2021:        ", (df["lastRefurbish"] > 2021).sum())
print("Floor above 50 (e.g. 999):     ", (df["floor"] > 50).sum())
print("Floor higher than the building:", (df["floor"] > df["numberOfFloors"]).sum())

Rent is 0 or below:             89
Rent above 10,000 €:            29
Size below 10 m²:               132
Size above 400 m²:              56
More than 20 rooms:             27
Built before 1800 or after 2021: 895
Refurbished after 2021:         10
Floor above 50 (e.g. 999):      28
Floor higher than the building: 1411


"Most problems affect only a small number of flats (fewer than X each), so removing or fixing them will not hurt the dataset."

Step 1: Investigate first. Is it really an error?
When many rows share the same "wrong" value, it's often not a typo but a different meaning:

What you see	Possible real meaning
Many rents = 0	"Price on request", so the rent is unknown, not free
Many sizes look 100× too big	Entered in a different unit
Many floors = 999	The form's code for "not stated"
Typos are random and rare. A large, repeated pattern usually has a reason. Look at the rows (df[condition].head()) before deciding.

Step 2: Choose an action
Situation	Action	Example
A systematic mistake you understand	Fix it	Divide by 100 if the unit was wrong
The value really means "unknown"	Turn it into missing (NaN), then use your missing plan (model / median / unknown / flag)	999 → NaN, then fill like other gaps
Most of the column is bad	Drop the column	As we did with mostly-empty columns
Only a few rows are bad	Remove those rows	Our case
The key idea: an impossible value is really a missing value in disguise. Turn it into NaN, and it gets handled by the plan you already made.


# Example: treat 999 as "unknown" instead of deleting the rows
df.loc[df["floor"] > 50, "floor"] = np.nan
(df.loc[rows, column] = value means: in these rows, in this column, set the value.)

Step 3: The special case of target columns
For baseRent and livingSpace we can't fill in a guess, because they create the target. A guessed rent would give a made-up answer for the model to learn from.

So rows with bad rent or size must be removed, even if there are many. If that would remove a large part of the data:

check whether the problem affects some groups more (for example, one region), because the remaining data could become biased
write it down as a limitation in the report
In short
Few bad rows → remove them.
Many bad rows → investigate, then fix or turn into NaN.
Target columns → can't guess, so remove the rows and report it if it's a lot.


### 3.5 Outliers (IQR rule)
Outlier = a rare but possibly real value. Normal range = Q1 − 1.5×IQR to Q3 + 1.5×IQR.


In [ ]:
# A function that finds outliers in ONE column using the IQR rule
def iqr_outliers(col):
    q1 = df[col].quantile(0.25)        # 25% of values are below this
    q3 = df[col].quantile(0.75)        # 75% of values are below this
    iqr = q3 - q1                      # spread of the middle half
    low = q1 - 1.5 * iqr               # lower limit of "normal"
    high = q3 + 1.5 * iqr              # upper limit of "normal"
    n = ((df[col] < low) | (df[col] > high)).sum()   # count values outside the limits
    print(f"{col}: normal range {low:,.0f} to {high:,.0f} → {n:,} outliers ({n / len(df):.1%})")

# Run it on the most important number columns
for col in ["baseRent", "livingSpace", "noRooms", "yearConstructed"]:
    iqr_outliers(col)


baseRent: normal range -354 to 1,490 → 14,458 outliers (5.4%)
livingSpace: normal range 4 to 136 → 11,459 outliers (4.3%)
noRooms: normal range 0 to 4 → 8,061 outliers (3.0%)
yearConstructed: normal range 1,881 to 2,065 → 3,212 outliers (1.2%)


**Decision on outliers:** keep them.
The IQR rule flags many real flats: expensive or large flats in major cities (baseRent, livingSpace), ordinary 5-room flats (noRooms), and historic buildings (yearConstructed). These are rare but real, and deleting them would remove genuine High-rent flats from the target.
Only the impossible values from step 3.4 will be removed or fixed in Phase 4.
**Lesson:** the IQR rule is an alarm that points where to look, not proof of an error.
You're right about rent and space: high values in Munich or Frankfurt are real flats, not errors. If we deleted them, we'd remove many of the genuinely expensive flats, and the High class would lose exactly the flats that define it.

For rooms and year, though, the IQR rule is also misleading. Look at what it does:

Column	Q1	Q3	IQR	"Normal" range	What gets flagged
noRooms	2	3	1	0.5 to 4.5	Every flat with 5 or more rooms: ordinary family flats ❌
yearConstructed	1950	1996	46	1881 to 2065	Every building before 1881: real historic buildings ❌


Remove what's impossible. Keep what's rare.



### 3.6 Text and encoding problems

In [ ]:
# Text columns we plan to KEEP as model inputs
text_cols = ["regio1", "regio2", "regio3", "heatingType", "condition",
             "interiorQual", "petsAllowed", "typeOfFlat"]

for col in text_cols:
    # .str says: "apply this text tool to every value in the Series, one by one":
    values = df[col].dropna().astype(str)                       # remove empty cells, make sure it's text
    html   = values.str.contains("&[a-z]+;", regex=True).sum()  # HTML codes like &uuml;
    broken = values.str.contains("Ã").sum()                     # garbled letters like Ã¼
    print(f"{col}: HTML codes = {html}, broken letters = {broken}")



regio1: HTML codes = 0, broken letters = 0
regio2: HTML codes = 0, broken letters = 0
regio3: HTML codes = 0, broken letters = 0
heatingType: HTML codes = 0, broken letters = 0
condition: HTML codes = 0, broken letters = 0
interiorQual: HTML codes = 0, broken letters = 0
petsAllowed: HTML codes = 0, broken letters = 0
typeOfFlat: HTML codes = 0, broken letters = 0


#### 📝 Note: `.str` and `.dropna()`

**`.str` (text accessor)**
- `.str` belongs to a pandas **Series**. It is not a global function.
- It applies a text tool to **every value** in the Series, one by one.
- `"München".lower()` works on ONE text; `df["regio2"].str.lower()` works on ALL values.
- Common tools: `.str.lower()`, `.str.contains("x")`, `.str.replace("a", "b")`, `.str.len()`.
- ⚠️ Only works on **text** columns, not numbers.

**`.dropna()`: removes empty values (NaN)**
| Used on | What it removes |
|---|---|
| A **Series** (one column) | The empty **cells** (each row is one cell) |
| A **DataFrame** (table) | Every **row** that has an empty cell in ANY column ⚠️ (would delete most of our data) |

**Does it change `df`?** No. It returns a **new copy**. `df` only changes if we save over it (`df = df.dropna()`).

**Why we used it in 3.6:** only to check the real text values. We made a temporary copy without empty cells, checked it, and `df` stayed unchanged, so it still has its empty cells.

**When are the empty cells handled?** In **Phase 4**, using the missing-value plan from step 3.2 (drop / "unknown" / flag / median / model).

> **Phase 3 = look and decide. Phase 4 = act.** `df` only changes in Phase 4.

### 3.7 Rare categories
A category is "rare" if it appears in fewer than 1% of rows. we need to deal with them so if they appeared in the test set the model can deal with them


In [42]:
# Category columns we plan to keep
cat_cols = ["regio1", "regio2", "regio3", "heatingType", "condition",
            "interiorQual", "petsAllowed", "typeOfFlat"]

for col in cat_cols:
    shares = df[col].value_counts(normalize=True) * 100   # % of rows for each category
    rare = shares[shares < 1]                             # keep only categories under 1%
    print(f"{col}: {df[col].nunique()} categories, {len(rare)} of them rare (<1%)")

regio1: 16 categories, 1 of them rare (<1%)
regio2: 419 categories, 402 of them rare (<1%)
regio3: 8684 categories, 8682 of them rare (<1%)
heatingType: 13 categories, 6 of them rare (<1%)
condition: 10 categories, 2 of them rare (<1%)
interiorQual: 4 categories, 0 of them rare (<1%)
petsAllowed: 3 categories, 0 of them rare (<1%)
typeOfFlat: 10 categories, 2 of them rare (<1%)


**Findings and decisions (rare categories):**
- **`regio3` (neighbourhood):** ~8,700 categories, most with very few flats, too detailed to learn from → **drop**.
- **`regio2` (city/district):** 419 categories. Many look rare as a %, but large cities have thousands of flats, and location is likely the strongest price clue → **keep**, and group districts with fewer than 500 flats into `"other"`.
- **`condition`:** merge by meaning (`ripe_for_demolition` → `need_of_renovation`, `negotiable` → `unknown`).
- **Other category columns:** group rare values into `"other"`.

**Lesson:** on large data, "rare" should be judged by **count** (e.g. fewer than 500 rows), not by %; 1% of 268,850 is already 2,688 flats.

In [43]:
# A category is "rare" if it has fewer than this many flats
MIN_COUNT = 500

# Category columns we plan to keep
cat_cols = ["regio1", "regio2", "regio3", "heatingType", "condition",
            "interiorQual", "petsAllowed", "typeOfFlat"]

for col in cat_cols:
    counts = df[col].value_counts()            # number of flats in each category
    rare = counts[counts < MIN_COUNT]          # keep only categories with too few flats
    print(f"{col}: {len(counts)} categories, {len(rare)} rare (<{MIN_COUNT} flats), "
          f"covering {rare.sum():,} flats")


regio1: 16 categories, 0 rare (<500 flats), covering 0 flats
regio2: 419 categories, 286 rare (<500 flats), covering 60,217 flats
regio3: 8684 categories, 8642 rare (<500 flats), covering 227,929 flats
heatingType: 13 categories, 2 rare (<500 flats), covering 510 flats
condition: 10 categories, 1 rare (<500 flats), covering 4 flats
interiorQual: 4 categories, 0 rare (<500 flats), covering 0 flats
petsAllowed: 3 categories, 0 rare (<500 flats), covering 0 flats
typeOfFlat: 10 categories, 0 rare (<500 flats), covering 0 flats


In [44]:
counts = df["regio2"].value_counts()
print("Districts kept as themselves:", (counts >= MIN_COUNT).sum())
print("Districts grouped into 'other':", (counts < MIN_COUNT).sum())

Districts kept as themselves: 133
Districts grouped into 'other': 286


it is not save to treat large amout of data in regio2 as other because it waste valuable details the solution is by #### 📝 Note: handling a category with many values (high cardinality)
**Problem:** `regio2` has 419 districts. 286 have fewer than 500 flats, which is too few to learn from, but together they cover 60,217 flats (~22%).
Putting them all into ONE "other" group would mix cheap and expensive regions and erase location information for 22% of the data.

**Solution: group by a parent level.** Each rare district becomes `"other_" + state` (e.g. `other_Bayern`).
- Large districts (e.g. München, Berlin) keep their own category.
- Rare districts still keep their **state**, so regional price differences survive.


Method	Level	Used for
Group rare values into "other"	Basic	Very common, in almost every project
Group rare values by a parent group (like the state)	Intermediate	Common when the data has a natural hierarchy: city → state, product → category
Replace each category with a number learned from the data (such as its typical price)	Advanced	Common in competitions and industry, but easy to get wrong (leakage)

In [ ]:
# Step 1: number of flats in each district
counts = df["regio2"].value_counts()

# Step 2: names of the districts with too few flats
rare_districts = counts[counts < MIN_COUNT].index

# Which flats are in a rare district? (True / False)
is_rare = df["regio2"].isin(rare_districts)

# Where the district is rare → replace it with "other_" + state. Otherwise keep it.
regio2_new = df["regio2"].mask(is_rare, "other_" + df["regio1"])

# Step 7: check the result
new_counts = regio2_new.value_counts()
print("Districts before:      ", df["regio2"].nunique())
print("Rare districts grouped:", len(rare_districts))
print("Categories after:      ", new_counts.size)
print("Smallest group now:    ", new_counts.min(), "flats")

# The smallest groups, to check that each one has enough flats
new_counts.tail(10)

Districts before:       419
Rare districts grouped: 286
Categories after:       146
Smallest group now:     296 flats


regio2
Regensburg                  524
Heidelberg                  522
Brandenburg_an_der_Havel    518
Mainz_Bingen_Kreis          517
Heilbronn_Kreis             515
Karlsruhe_Kreis             511
Barnim_Kreis                504
Koblenz                     503
Salzgitter                  501
other_Sachsen               296
Name: count, dtype: int64

.mask() vs .where()
They're opposites:

Method	Replaces the value when the condition is…
.mask(condition, new)	True
.where(condition, new)	False